In [1]:
import pygame, csv, time, random, os, json

# --- CONFIGURATION ---
SCREEN_WIDTH = 1400
SCREEN_HEIGHT = 800
CURSOR_RADIUS = 22
TARGET_RADIUS = 14
FPS = 60
FILE_NAME = "experiment_data.csv"
RECORDING_DIR = "recordings"
NUM_ITERATIONS = 20
# Colors
WHITE = (255,255,255)
BLACK = (0,0,0)
RED = (255,0,0)
BLUE = (0,0,255)
GRAY = (128,128,128)
# Game Modes
INDIVIDUAL = "individual"
COOPERATIVE = "cooperative"
PLAYBACK = "playback"
AI = "ai"
# AI Control modes
AI_CONTROLS_VERTICAL = 1
AI_CONTROLS_HORIZONTAL = 0
# Preset target locations for 20 iterations, scaled for 1400 x 800
PRESET_TARGETS = [
    [250,120], [1150,120], [700,170], [350,300], [1050,300],
    [180,450], [1220,450], [1300,120], [950,560], [520,620],
    [780,120], [950,170], [350,250], [1150,250], [220,650],
    [260,540], [1200,620], [420,680], [1030,650], [650,700]
]
class ExperimentGame:
    def __init__(self, mode=INDIVIDUAL, recording_file=None, iteration=1, ai_axis=None, target_pos=None, playback_axis=None):
        pygame.init()
        self.screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
        pygame.display.set_caption("CogSci Joint Action Task")
        self.clock = pygame.time.Clock()
        self.running = True
        self.mode = mode
        self.iteration = iteration
        self.target_hit = False
        # Positions
        self.cursor_pos = [SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2]
        self.target_pos = target_pos if target_pos is not None else PRESET_TARGETS[iteration - 1]
        # Velocity summing variables
        self.human_v = [0,0]
        self.partner_v = [0,0]
        # Data Logging Setup
        self.data_log = []
        self.start_time = time.time()
        self.recording_file = recording_file
        # Playback Setup
        self.playback_data = {}
        self.playback_index = 0
        self.playback_axis = playback_axis
        self.recording_target_pos = None
        if mode == PLAYBACK and recording_file:
            self.load_recording(recording_file)
        # AI Setup
        self.ai_control_axis = ai_axis if ai_axis is not None else AI_CONTROLS_VERTICAL
    def load_recording(self, filename):
        """Load pre-recorded movements from a JSON file"""
        try:
            with open(filename, 'r') as f:
                recording_data = json.load(f)
            if 'iterations' in recording_data:
                iterations = recording_data.get('iterations', [])
                converted_iterations = []
                for iter_data in iterations:
                    frames = iter_data.get('frames', [])
                    if frames:
                        partner_horizontal = [f['partner_vx'] for f in frames]
                        partner_vertical = [f['partner_vy'] for f in frames]
                    else:
                        partner_horizontal = iter_data.get('partner_horizontal', [])
                        partner_vertical = iter_data.get('partner_vertical', [])
                    converted_iterations.append({
                        'iteration': iter_data.get('iteration', 0),
                        'target_pos': iter_data['target_pos'],
                        'partner_horizontal': partner_horizontal,
                        'partner_vertical': partner_vertical,
                        'horizontal': partner_horizontal,
                        'vertical': partner_vertical
                    })
                self.playback_data = {'type': 'session', 'iterations': converted_iterations}
                if 1 <= self.iteration <= len(converted_iterations):
                    self.recording_target_pos = converted_iterations[self.iteration - 1].get('target_pos', self.target_pos)
                    self.target_pos = self.recording_target_pos
                else:
                    self.recording_target_pos = self.target_pos
                print(f"Loaded session recording with {len(converted_iterations)} iterations")
            else:
                self.recording_target_pos = recording_data.get('target_pos', self.target_pos)
                partner_horizontal = recording_data.get('partner_horizontal', recording_data.get('horizontal', []))
                partner_vertical = recording_data.get('partner_vertical', recording_data.get('vertical', []))
                self.playback_data = {
                    'type': 'single',
                    'partner_horizontal': partner_horizontal,
                    'partner_vertical': partner_vertical,
                    'horizontal': partner_horizontal,
                    'vertical': partner_vertical,
                    'target_pos': self.recording_target_pos
                }
                print(f"Loaded recording with {len(partner_horizontal)} horizontal and {len(partner_vertical)} vertical frames")
        except FileNotFoundError:
            print(f"Recording file {filename} not found.")
            self.playback_data = {'type': 'single', 'horizontal': [], 'vertical': [], 'target_pos': self.target_pos}
    def get_ai_input(self):
        """AI agent that controls either horizontal or vertical movement"""
        dx = self.target_pos[0] - self.cursor_pos[0]
        dy = self.target_pos[1] - self.cursor_pos[1]
        # AI strength
        k = 0.12
        if self.ai_control_axis == AI_CONTROLS_VERTICAL:
            return [0, dy * k]
        else:
            return [dx * k, 0]
    def get_playback_input(self):
        """Get pre-recorded movement for current frame"""
        if self.playback_axis is None:
            return [0,0]
        if self.playback_data.get('type') == 'session':
            iterations = self.playback_data.get('iterations', [])
            if 1 <= self.iteration <= len(iterations):
                current_axis = iterations[self.iteration - 1].get(f'partner_{self.playback_axis}', [])
            else:
                current_axis = []
        else:
            current_axis = self.playback_data.get(f'partner_{self.playback_axis}', self.playback_data.get(self.playback_axis, []))
        if self.playback_index < len(current_axis):
            velocity = current_axis[self.playback_index]
            self.playback_index += 1
            if self.playback_axis == 'horizontal':
                return [velocity, 0]
            else:
                return [0, velocity]
        return [0,0]
    def log_frame(self):
        """Log frame data"""
        elapsed = time.time() - self.start_time
        self.data_log.append([
            elapsed,
            self.cursor_pos[0], self.cursor_pos[1],
            self.human_v[0], self.human_v[1],
            self.partner_v[0], self.partner_v[1],
            self.target_pos[0], self.target_pos[1],
            self.mode,
            self.iteration,
            self.ai_control_axis if self.mode == AI else "",
            self.playback_axis if self.mode == PLAYBACK else ""
        ])
    def save_data(self):
        """Save experiment data to CSV"""
        keys = ["timestamp","cursor_x","cursor_y","h_vx","h_vy","p_vx","p_vy","target_x","target_y","mode","iteration","ai_axis","playback_axis"]
        if not os.path.exists(FILE_NAME):
            with open(FILE_NAME, "w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(keys)
        with open(FILE_NAME, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerows(self.data_log)
        print(f"Data saved to {FILE_NAME}")
    def draw_text(self, text, font, color, surface, x, y):
        """Draw text on the screen"""
        textobj = font.render(text, True, color)
        textrect = textobj.get_rect()
        textrect.topleft = (x, y)
        surface.blit(textobj, textrect)
    def run(self):
        """Run one iteration - ends when target is hit"""
        font = pygame.font.Font(None, 24)
        speed = 20
        while self.running and not self.target_hit:
            self.screen.fill(WHITE)
            # Reset movement every frame
            self.human_v = [0,0]
            self.partner_v = [0,0]
            # 1. Event Handling
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    self.running = False
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_ESCAPE:
                        self.running = False
                    # INDIVIDUAL MODE
                    if self.mode == INDIVIDUAL:
                        if event.key == pygame.K_LEFT:
                            self.human_v[0] = -speed
                        elif event.key == pygame.K_RIGHT:
                            self.human_v[0] = speed
                        elif event.key == pygame.K_UP:
                            self.human_v[1] = -speed
                        elif event.key == pygame.K_DOWN:
                            self.human_v[1] = speed
                    # COOPERATIVE MODE
                    elif self.mode == COOPERATIVE:
                        if event.key == pygame.K_LEFT:
                            self.human_v[0] = -speed
                        elif event.key == pygame.K_RIGHT:
                            self.human_v[0] = speed
                        elif event.key == pygame.K_w:
                            self.partner_v[1] = -speed
                        elif event.key == pygame.K_s:
                            self.partner_v[1] = speed
                    # PLAYBACK MODE
                    elif self.mode == PLAYBACK:
                        if self.playback_axis == 'horizontal':
                            if event.key == pygame.K_w:
                                self.human_v[1] = -speed
                            elif event.key == pygame.K_s:
                                self.human_v[1] = speed
                        else:
                            if event.key == pygame.K_LEFT:
                                self.human_v[0] = -speed
                            elif event.key == pygame.K_RIGHT:
                                self.human_v[0] = speed
                    # AI MODE
                    elif self.mode == AI:
                        if self.ai_control_axis == AI_CONTROLS_VERTICAL:
                            if event.key == pygame.K_LEFT:
                                self.human_v[0] = -speed
                            elif event.key == pygame.K_RIGHT:
                                self.human_v[0] = speed
                        else:
                            if event.key == pygame.K_w:
                                self.human_v[1] = -speed
                            elif event.key == pygame.K_s:
                                self.human_v[1] = speed
            # Playback partner movement
            if self.mode == PLAYBACK:
                self.partner_v = self.get_playback_input()
            # AI partner movement: AI acts only after participant acts
            elif self.mode == AI:
                if self.human_v != [0,0]:
                    self.partner_v = self.get_ai_input()
                else:
                    self.partner_v = [0,0]
            # 2. Joint Action: SUM THE VELOCITIES
            self.cursor_pos[0] += self.human_v[0] + self.partner_v[0]
            self.cursor_pos[1] += self.human_v[1] + self.partner_v[1]
            # Clamp cursor to screen bounds
            self.cursor_pos[0] = max(CURSOR_RADIUS, min(SCREEN_WIDTH - CURSOR_RADIUS, self.cursor_pos[0]))
            self.cursor_pos[1] = max(CURSOR_RADIUS, min(SCREEN_HEIGHT - CURSOR_RADIUS, self.cursor_pos[1]))
            # 3. Check Target Collision
            dist = ((self.cursor_pos[0] - self.target_pos[0]) ** 2 + (self.cursor_pos[1] - self.target_pos[1]) ** 2) ** 0.5
            if dist < (CURSOR_RADIUS + TARGET_RADIUS):
                self.target_hit = True
                print(f"Target Hit! Iteration {self.iteration} complete.")
            # 4. Draw everything
            pygame.draw.circle(self.screen, RED, self.target_pos, TARGET_RADIUS)
            pygame.draw.circle(self.screen, BLUE, (int(self.cursor_pos[0]), int(self.cursor_pos[1])), CURSOR_RADIUS)
            # Draw mode indicator and iteration
            mode_text = f"Mode: {self.mode.upper()} | Iteration: {self.iteration}/{NUM_ITERATIONS}"
            self.draw_text(mode_text, font, BLACK, self.screen, 10, 10)
            # Draw controls info
            if self.mode == INDIVIDUAL:
                self.draw_text("Controls: Arrow Keys", font, BLACK, self.screen, 10, 35)
            elif self.mode == COOPERATIVE:
                self.draw_text("P1: LEFT/RIGHT arrows | P2: W/S keys", font, BLACK, self.screen, 10, 35)
            elif self.mode == PLAYBACK:
                if self.playback_axis == 'horizontal':
                    self.draw_text("Playback: LEFT/RIGHT | You: W/S", font, BLACK, self.screen, 10, 35)
                else:
                    self.draw_text("Playback: UP/DOWN | You: LEFT/RIGHT", font, BLACK, self.screen, 10, 35)
            elif self.mode == AI:
                ai_axis = "UP/DOWN" if self.ai_control_axis == AI_CONTROLS_VERTICAL else "LEFT/RIGHT"
                player_axis = "LEFT/RIGHT" if self.ai_control_axis == AI_CONTROLS_VERTICAL else "UP/DOWN"
                if self.iteration == 1:
                    self.draw_text(f"AI block 1: AI controls {ai_axis} | You control {player_axis}", font, BLACK, self.screen, 10, 35)
                elif self.iteration == (NUM_ITERATIONS // 2) + 1:
                    self.draw_text(f"AXIS CHANGE: AI now controls {ai_axis} | You now control {player_axis}", font, BLACK, self.screen, 10, 35)
                else:
                    self.draw_text(f"AI controls: {ai_axis} | You control: {player_axis}", font, BLACK, self.screen, 10, 35)
            self.draw_text("Press ESC to quit", font, BLACK, self.screen, 10, 60)
            # 5. Log and Update
            self.log_frame()
            pygame.display.flip()
            self.clock.tick(FPS)
        if self.target_hit:
            self.save_data()
def save_session_recording(mode, session_records):
    """Save a full gameplay session as one JSON file with all iterations"""
    if not os.path.exists(RECORDING_DIR):
        os.makedirs(RECORDING_DIR)
    session_data = {
        'mode': mode,
        'num_iterations': len(session_records),
        'timestamp': int(time.time()),
        'iterations': session_records
    }
    filename = os.path.join(RECORDING_DIR, f"session_recording_{int(time.time())}.json")
    with open(filename, 'w') as f:
        json.dump(session_data, f, indent=2)
    print(f"Session recording saved to {filename}")
    return filename
def show_menu():
    """Display menu to select game mode"""
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("CogSci Joint Action Task - Mode Selection")
    clock = pygame.time.Clock()
    font = pygame.font.Font(None, 72)
    small_font = pygame.font.Font(None, 42)
    selected = 0
    modes = [
        (INDIVIDUAL, "1. Individual Mode"),
        (COOPERATIVE, "2. Cooperative Mode"),
        (PLAYBACK, "3. Playback Mode"),
        (AI, "4. AI Mode")
    ]
    selecting_mode = True
    while selecting_mode:
        screen.fill(WHITE)
        title = font.render("Select Game Mode", True, BLACK)
        screen.blit(title, (SCREEN_WIDTH // 2 - title.get_width() // 2, 100))
        start_y = 250
        for i, (mode_key, mode_text) in enumerate(modes):
            color = BLUE if i == selected else BLACK
            prefix = ">>> " if i == selected else "    "
            text = small_font.render(prefix + mode_text, True, color)
            screen.blit(text, (SCREEN_WIDTH // 2 - text.get_width() // 2, start_y + i * 100))
        info_text = small_font.render(f"Each mode will run {NUM_ITERATIONS} times", True, GRAY)
        screen.blit(info_text, (SCREEN_WIDTH // 2 - info_text.get_width() // 2, 680))
        instructions = small_font.render("UP/DOWN = select | ENTER = confirm | ESC = exit", True, GRAY)
        screen.blit(instructions, (SCREEN_WIDTH // 2 - instructions.get_width() // 2, 740))
        pygame.display.flip()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return None
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_UP:
                    selected = (selected - 1) % len(modes)
                elif event.key == pygame.K_DOWN:
                    selected = (selected + 1) % len(modes)
                elif event.key == pygame.K_RETURN:
                    selecting_mode = False
                elif event.key == pygame.K_ESCAPE:
                    pygame.quit()
                    return None
        clock.tick(FPS)
    pygame.quit()
    return modes[selected][0]
def show_recording_menu():
    """Display menu to select a recording file for playback"""
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("CogSci Joint Action Task - Select Recording")
    clock = pygame.time.Clock()
    font = pygame.font.Font(None, 60)
    small_font = pygame.font.Font(None, 32)
    recordings = []
    if os.path.exists(RECORDING_DIR):
        recordings = [f for f in os.listdir(RECORDING_DIR) if f.endswith('.json')]
        recordings.sort(reverse=True)
    if not recordings:
        screen.fill(WHITE)
        text = small_font.render("No recordings found. Run COOPERATIVE mode first to record.", True, BLACK)
        screen.blit(text, (SCREEN_WIDTH // 2 - text.get_width() // 2, SCREEN_HEIGHT // 2))
        pygame.display.flip()
        pygame.time.wait(2000)
        pygame.quit()
        return None
    selected = 0
    selecting = True
    while selecting:
        screen.fill(WHITE)
        title = font.render("Select Recording to Play", True, BLACK)
        screen.blit(title, (SCREEN_WIDTH // 2 - title.get_width() // 2, 80))
        for i, recording in enumerate(recordings[:10]):
            color = BLUE if i == selected else BLACK
            prefix = ">>> " if i == selected else "    "
            text = small_font.render(prefix + recording, True, color)
            screen.blit(text, (120, 180 + i * 45))
        instructions = small_font.render("UP/DOWN = select | ENTER = confirm | ESC = cancel", True, GRAY)
        screen.blit(instructions, (SCREEN_WIDTH // 2 - instructions.get_width() // 2, 720))
        pygame.display.flip()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return None
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_UP:
                    selected = (selected - 1) % min(len(recordings), 10)
                elif event.key == pygame.K_DOWN:
                    selected = (selected + 1) % min(len(recordings), 10)
                elif event.key == pygame.K_RETURN:
                    selecting = False
                elif event.key == pygame.K_ESCAPE:
                    pygame.quit()
                    return None
        clock.tick(FPS)
    pygame.quit()
    return os.path.join(RECORDING_DIR, recordings[selected])
def show_iteration_ready_screen(mode, iteration, total, ai_axis=None, playback_axis=None):
    """Display control instructions before each iteration - wait for SPACE to start"""
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("CogSci Joint Action Task - Ready for Next Trial")
    clock = pygame.time.Clock()
    font = pygame.font.Font(None, 64)
    small_font = pygame.font.Font(None, 36)
    waiting = True
    cancelled = False
    while waiting:
        screen.fill(WHITE)
        iteration_text = font.render(f"Iteration {iteration} of {total}", True, BLACK)
        screen.blit(iteration_text, (SCREEN_WIDTH // 2 - iteration_text.get_width() // 2, 80))
        y_pos = 220
        controls_title = small_font.render("Controls:", True, BLACK)
        screen.blit(controls_title, (SCREEN_WIDTH // 2 - 300, y_pos))
        y_pos += 50
        if mode == INDIVIDUAL:
            control_text = small_font.render("Use Arrow Keys to control the ball", True, BLACK)
            screen.blit(control_text, (SCREEN_WIDTH // 2 - control_text.get_width() // 2, y_pos))
        elif mode == COOPERATIVE:
            control_text1 = small_font.render("Player 1: LEFT/RIGHT Arrow Keys", True, BLACK)
            control_text2 = small_font.render("Player 2: W/S Keys", True, BLACK)
            screen.blit(control_text1, (SCREEN_WIDTH // 2 - control_text1.get_width() // 2, y_pos))
            screen.blit(control_text2, (SCREEN_WIDTH // 2 - control_text2.get_width() // 2, y_pos + 45))
        elif mode == PLAYBACK:
            if playback_axis == 'horizontal':
                control_text1 = small_font.render("Playback controls: LEFT/RIGHT", True, BLACK)
                control_text2 = small_font.render("You control: W/S", True, BLACK)
            else:
                control_text1 = small_font.render("Playback controls: UP/DOWN", True, BLACK)
                control_text2 = small_font.render("You control: LEFT/RIGHT", True, BLACK)
            screen.blit(control_text1, (SCREEN_WIDTH // 2 - control_text1.get_width() // 2, y_pos))
            screen.blit(control_text2, (SCREEN_WIDTH // 2 - control_text2.get_width() // 2, y_pos + 45))
        elif mode == AI:
            ai_control = "UP/DOWN" if ai_axis == AI_CONTROLS_VERTICAL else "LEFT/RIGHT"
            player_control = "LEFT/RIGHT" if ai_axis == AI_CONTROLS_VERTICAL else "UP/DOWN"
            if iteration == 1:
                control_text0 = small_font.render("AI Block 1", True, BLUE)
            elif iteration == (total // 2) + 1:
                control_text0 = small_font.render("AXIS CHANGE", True, BLUE)
            else:
                control_text0 = small_font.render("AI Condition", True, BLUE)
            control_text1 = small_font.render(f"AI controls: {ai_control}", True, BLACK)
            control_text2 = small_font.render(f"You control: {player_control}", True, BLACK)
            screen.blit(control_text0, (SCREEN_WIDTH // 2 - control_text0.get_width() // 2, y_pos - 50))
            screen.blit(control_text1, (SCREEN_WIDTH // 2 - control_text1.get_width() // 2, y_pos))
            screen.blit(control_text2, (SCREEN_WIDTH // 2 - control_text2.get_width() // 2, y_pos + 45))
        goal_text = small_font.render("Goal: Move the BLUE ball into the RED target", True, BLACK)
        screen.blit(goal_text, (SCREEN_WIDTH // 2 - goal_text.get_width() // 2, 460))
        ready_text = font.render("Press SPACE to start", True, BLUE)
        screen.blit(ready_text, (SCREEN_WIDTH // 2 - ready_text.get_width() // 2, 560))
        esc_text = small_font.render("Press ESC to quit and return to menu", True, GRAY)
        screen.blit(esc_text, (SCREEN_WIDTH // 2 - esc_text.get_width() // 2, 680))
        pygame.display.flip()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return False
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE:
                    waiting = False
                elif event.key == pygame.K_ESCAPE:
                    cancelled = True
                    waiting = False
        clock.tick(FPS)
    pygame.quit()
    return not cancelled
if __name__ == "__main__":
    while True:
        mode = show_menu()
        if mode is None:
            print("Exiting game.")
            break
        recording_file = None
        if mode == PLAYBACK:
            recording_file = show_recording_menu()
            if recording_file is None:
                print("No recording selected. Returning to menu.")
                continue
        target_order = list(range(NUM_ITERATIONS))
        random.shuffle(target_order)
        session_records = []
        user_cancelled = False
        for iteration in range(1, NUM_ITERATIONS + 1):
            ai_axis = None
            if mode == AI:
                if iteration <= NUM_ITERATIONS / 2:
                    ai_axis = AI_CONTROLS_VERTICAL
                else:
                    ai_axis = AI_CONTROLS_HORIZONTAL
            playback_axis = None
            if mode == PLAYBACK:
                if iteration <= NUM_ITERATIONS / 2:
                    playback_axis = 'vertical'
                else:
                    playback_axis = 'horizontal'
            target_pos = PRESET_TARGETS[target_order[iteration - 1]]
            if not show_iteration_ready_screen(mode, iteration, NUM_ITERATIONS, ai_axis=ai_axis, playback_axis=playback_axis):
                print(f"User cancelled during iteration {iteration}. Returning to menu.")
                user_cancelled = True
                break
            game = ExperimentGame(mode=mode, recording_file=recording_file, iteration=iteration, ai_axis=ai_axis, target_pos=target_pos, playback_axis=playback_axis)
            game.run()
            if not game.running:
                print(f"User cancelled during iteration {iteration}. Returning to menu.")
                user_cancelled = True
                break
            if mode == COOPERATIVE:
                session_records.append({
                    'iteration': iteration,
                    'target_pos': game.target_pos,
                    'human_horizontal': [entry[3] for entry in game.data_log],
                    'human_vertical': [entry[4] for entry in game.data_log],
                    'partner_horizontal': [entry[5] for entry in game.data_log],
                    'partner_vertical': [entry[6] for entry in game.data_log],
                    'horizontal': [entry[5] for entry in game.data_log],
                    'vertical': [entry[6] for entry in game.data_log]
                })
        if not user_cancelled:
            if mode == COOPERATIVE and session_records:
                recording_file = save_session_recording(mode, session_records)
                print(f"\nSession recording saved: {recording_file}")
            print(f"\nCompleted all {NUM_ITERATIONS} iterations of {mode.upper()} mode!")
            print("Returning to menu...")

SyntaxError: invalid syntax (346411928.py, line 1)